<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 7: </b>Vector Store를 이용한 Retrieval-Augmented Generation</h2>
<br>

이전 노트북에서는 임베딩 모델에 대해 배우고 그 기능 일부를 연습했습니다. 긴 형식의 문서 비교라는 본래 용도를 논의하고, 더 커스텀한 시맨틱 비교의 근간으로 활용하는 방법을 찾았습니다. 이 노트북에서는 이 아이디어를 검색 모델의 본래 용도로 발전시키고, 정보를 자동으로 저장하고 검색하기 위해 *vector store*에 의존하는 챗봇 시스템을 구축하는 방법을 탐구합니다.

<br>

### **학습 목표:**

- 시맨틱 유사도 기반 시스템이 어떻게 사용하기 쉬운 검색 방식을 가능하게 하는지 이해합니다.

- 검색 모듈을 채팅 모델 시스템에 통합하여 retrieval-augmented generation(RAG) 파이프라인을 만드는 방법을 배웁니다. 이는 문서 검색과 대화 메모리 버퍼 같은 작업에 적용할 수 있습니다.

<br>

### **생각해 볼 질문:**

- 이 노트북은 계층적 추론이나 (플래닝 에이전트 같은) 비단순 RAG를 통합하려 하지 않습니다. 이러한 구성 요소가 LCEL 체인에서 동작하려면 어떤 수정이 필요할지 생각해 보세요.

- vector store 솔루션을 확장 가능한 서비스로 옮기는 것이 언제 가장 좋을지, 그리고 최적화를 위해 GPU가 언제 필요해질지 생각해 보세요.

<br>

### **환경 설정:**

In [ ]:
# %%capture
## ^^ Comment out if you want to see the pip install process

## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints fastembed gradio rich
# %pip install -q arxiv pymupdf faiss-cpu
# !mkdir -p cached_papers && test -s cached_papers/2210.03629v3.pdf || wget -q --tries=3 --timeout=20 -O cached_papers/2210.03629v3.pdf https://arxiv.org/pdf/2210.03629v3

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

embedder = NVIDIAEmbeddings(
    model="course/embedding",
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

----

<br>

## Part 1: RAG 워크플로 요약

이 노트북은 여러 패러다임을 탐구하고, 가장 흔한 retrieval-augmented 워크플로에 접근하는 데 도움이 되는 참고 코드를 도출합니다. 구체적으로 다음 섹션들을 다룹니다(차이점은 강조 표시):

<br>

> ***대화 교환을 위한 Vector Store 워크플로:***
- 새로운 대화마다 시맨틱 임베딩을 생성합니다.
- 메시지 본문을 검색을 위해 vector store에 추가합니다.
- vector store에서 관련 메시지를 조회하여 LLM 컨텍스트를 채웁니다.

<br>

> ***임의의 문서를 위한 수정된 워크플로:***
- **문서를 청크로 나누고 유용한 메시지로 가공합니다.**
- **새로운 문서 청크**마다 시맨틱 임베딩을 생성합니다.
- **청크 본문**을 검색을 위해 vector store에 추가합니다.
- vector store에서 관련 **청크**를 조회하여 LLM 컨텍스트를 채웁니다.
    - ***선택:* 더 나은 LLM 결과를 위해 결과를 수정/합성합니다.**

<br>

> **임의의 문서 디렉터리를 위한 확장 워크플로:**
- **각 문서**를 청크로 나누고 유용한 메시지로 가공합니다.
- 새로운 문서 청크마다 시맨틱 임베딩을 생성합니다.
- 청크 본문을 **빠른 검색을 위한 확장 가능한 벡터 데이터베이스**에 추가합니다.
    - ***선택*: 더 큰 시스템을 위해 계층 구조나 메타데이터 구조를 활용합니다.**
- **벡터 데이터베이스**에서 관련 청크를 조회하여 LLM 컨텍스트를 채웁니다.
    - *선택:* 더 나은 LLM 결과를 위해 결과를 수정/합성합니다.

<br>

RAG와 관련된 가장 중요한 용어 일부는 [**LlamaIndex Concepts 페이지**](https://developers.llamaindex.ai/python/framework/getting_started/concepts/)에 자세히 다루어져 있으며, 이 페이지 자체가 LlamaIndex의 로딩 및 검색 전략으로 나아가기 위한 훌륭한 출발점입니다. 이 노트북을 진행하면서 참고 자료로 활용하기를 강력히 권장하며, 코스 후에 LlamaIndex를 직접 사용해 보고 장단점을 몸소 느껴 보시길 권합니다!

<!-- > <img src="https://drive.google.com/uc?export=view&id=1cFbKbVvLLnFPs3yWCKIuzXkhBWh6nLQY" width=1200px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/data_connection_langchain.jpeg" width=1200px/>
>
> 출처: [**Retrieval | LangChain**🦜️🔗](https://blog.langchain.com/syncing-data-sources-to-vector-stores/)

----

<br>

## **Part 2:** 대화 기록을 위한 RAG

이전 탐구에서는 문서 임베딩 모델의 기능을 파고들어, 텍스트의 시맨틱 벡터 표현을 임베딩하고 저장하고 비교하는 데 사용했습니다. 이를 수동으로 vector store 영역으로 효율적으로 확장하는 방법을 설명할 수도 있지만, 표준 API를 사용하는 진정한 아름다움은 이미 무거운 일을 대신해 줄 수 있는 다른 프레임워크와의 강력한 통합에 있습니다!

<br>

### **Step 1**: 대화 얻기

Llama-13B로 만들어진, 채팅 에이전트와 Beras라는 파란 곰 사이의 대화를 생각해 봅시다. 세부 사항과 잠재적 곁가지가 가득한 이 대화는 우리 연구를 위한 풍부한 데이터셋을 제공합니다:

In [ ]:
conversation = [  ## This conversation was generated partially by an AI system, and modified to exhibit desirable properties
    "[User]  Hello! My name is Beras, and I'm a big blue bear! Can you please tell me about the rocky mountains?",
    "[Agent] The Rocky Mountains are a beautiful and majestic range of mountains that stretch across North America",
    "[Beras] Wow, that sounds amazing! Ive never been to the Rocky Mountains before, but Ive heard many great things about them.",
    "[Agent] I hope you get to visit them someday, Beras! It would be a great adventure for you!",
    "[Beras] Thank you for the suggestion! Ill definitely keep it in mind for the future.",
    "[Agent] In the meantime, you can learn more about the Rocky Mountains by doing some research online or watching documentaries about them.",
    "[Beras] I live in the arctic, so I'm not used to the warm climate there. I was just curious, ya know!",
    "[Agent] Absolutely! Lets continue the conversation and explore more about the Rocky Mountains and their significance!"
]

이전 노트북의 수동 임베딩 전략도 여전히 충분히 쓸 만하지만, 마음 편히 **vector store**가 그 모든 작업을 대신하게 할 수도 있습니다!

<br>

### **Step 2:** Vector Store Retriever 구성하기

대화에 대한 유사도 쿼리를 간소화하기 위해, 문단을 대신 추적해 주는 vector store를 사용할 수 있습니다! **Vector Store**, 즉 벡터 저장 시스템은 임베딩/비교 전략의 저수준 세부 사항 대부분을 추상화하고, 벡터를 로드하고 비교하는 간단한 인터페이스를 제공합니다.

<!-- > <img src="https://drive.google.com/uc?export=view&id=1ZjwYbSZzsXK6ZP8O1-cY3BeRffV4oqzb" width=1000px/> -->
> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/vector_stores.jpeg" width=1200px/>
>
> 출처: [**Vector Stores | LangChain**🦜️🔗](https://blog.langchain.com/syncing-data-sources-to-vector-stores/vectorstores/)

<br>

API 관점에서 과정을 단순화하는 것 외에도, vector store는 내부적으로 커넥터, 통합, 최적화를 구현합니다. 우리는 [**FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss)부터 시작합니다. 이는 LangChain 호환 임베딩 모델을 [**FAISS(Facebook AI Similarity Search)**](https://github.com/facebookresearch/faiss) 라이브러리와 통합하여 로컬 머신에서 과정을 빠르고 확장 가능하게 만들어 줍니다!

**구체적으로:**

1. `from_texts` 생성자를 통해 대화를 [**FAISS vector store**](https://python.langchain.com/docs/integrations/vectorstores/faiss)에 넣을 수 있습니다. 이는 대화 데이터와 임베딩 모델을 받아 대화에 대한 검색 가능한 인덱스를 만듭니다.
2. 이 vector store는 retriever로 "해석"될 수 있으며, LangChain runnable API를 지원하고 입력 쿼리로 검색된 문서를 반환합니다.

다음은 LangChain `vectorstore` API를 사용해 FAISS vector store를 구성하고 이를 retriever로 재해석하는 방법을 보여 줍니다:

In [ ]:
%%time
## ^^ This cell will be timed to see how long the conversation embedding takes
from langchain_community.vectorstores import FAISS

## Streamlined from_texts FAISS vectorstore construction from text list
convstore = FAISS.from_texts(conversation, embedding=embedder)
retriever = convstore.as_retriever()

이제 retriever를 다른 LangChain runnable처럼 사용해 vector store에서 관련 문서를 조회할 수 있습니다:

In [ ]:
pprint(retriever.invoke("What is your name?"))

In [ ]:
pprint(retriever.invoke("Where are the Rocky Mountains?"))

보시다시피 retriever는 쿼리로부터 의미적으로 관련 있는 문서 몇 개를 찾았습니다. 모든 문서가 그 자체로 유용하거나 명확하지는 않다는 점을 눈치채셨을 겁니다. 예를 들어 *"your name"* 에 대해 *"Beras"* 가 검색되는 것은 맥락 없이 제공될 경우 챗봇에 문제가 될 수 있습니다. 잠재적 문제를 예상하고 LLM 구성 요소 간 시너지를 만들면 좋은 RAG 동작의 가능성을 높일 수 있으니, 이런 함정과 기회를 주의 깊게 살펴보세요.

<br>

### **선택 단계:** 검색된 후보 리랭킹(Reranking)

vector store는 압축된 표현을 비교하므로 많은 저장 문서를 검색하기에 실용적입니다. 작은 후보 집합이 만들어지면, cross-encoder가 쿼리를 각 후보와 함께 검토하여 순서를 정제할 수 있습니다. 이는 쿼리 시점에 작업을 추가하므로, 벡터 검색이 컬렉션을 좁힌 뒤에 가장 유용합니다.

이 두 번째 단계를 위해, 코스 모델 서버는 `cross-encoder/ms-marco-MiniLM-L6-v2`를 CPU에 로드해 두고 NVIDIA 랭킹 인터페이스를 통해 노출합니다. 따라서 `NVIDIARerank`는 호스팅되거나 별도로 배포된 NVIDIA NIM과 같은 클라이언트 경계를 사용합니다. Colab에서는 API 키를 제공한 뒤 모델과 base URL을 변경하여 호환되는 NVIDIA 리랭킹 엔드포인트를 대상으로 커넥터를 사용할 수 있습니다.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIARerank

rerank_query = "Where are the Rocky Mountains?"
candidates = retriever.invoke(rerank_query)
reranker = NVIDIARerank(
    model="course/reranker",
    base_url="http://llm_client:9000/v1",
    top_n=len(candidates),
)
reranked = reranker.compress_documents(candidates, query=rerank_query)

for rank, document in enumerate(reranked, start=1):
    score = document.metadata["relevance_score"]
    print(f"{rank}. {score:.3f}  {document.page_content}")

### **Step 3:** 대화 검색을 체인에 통합하기

로드된 retriever 구성 요소를 체인으로 갖추었으니, 이전처럼 기존 채팅 시스템에 통합할 수 있습니다. 구체적으로 다음과 같은 ***항상 켜진(always-on) RAG 방식***부터 시작할 수 있습니다:
- **retriever가 기본적으로 항상 컨텍스트를 검색합니다**.
- **generator가 검색된 컨텍스트를 바탕으로 동작합니다**.

In [ ]:
from langchain_community.document_transformers import LongContextReorder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from functools import partial
from operator import itemgetter

########################################################################
## Utility Runnables/Methods
def RPrint(preface=""):
    """Simple passthrough "prints, then returns" chain"""
    def print_and_return(x, preface):
        if preface: print(preface, end="")
        pprint(x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name:
            out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

## Optional; Reorders longer documents to center of output text
long_reorder = RunnableLambda(LongContextReorder().transform_documents)

In [ ]:
context_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {question}"
    "\nAnswer the user conversationally. User is not aware of context."
)

chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'question': (lambda x:x)
    }
    | context_prompt
    # | RPrint()
    | instruct_llm
    | StrOutputParser()
)

pprint(chain.invoke("Where does Beras live?"))

잠시 시간을 내어 몇 가지 호출을 더 시도해 보고 새로운 구성이 어떻게 동작하는지 확인하세요. 어떤 모델을 선택했든, 다음 질문들이 흥미로운 출발점이 될 것입니다.

In [ ]:
pprint(chain.invoke("Where are the Rocky Mountains?"))

In [ ]:
pprint(chain.invoke("Where are the Rocky Mountains? Are they close to California?"))

In [ ]:
pprint(chain.invoke("How far away is Beras from the Rocky Mountains?"))

<br>

LLM에 실제로 입력되는 컨텍스트가 비교적 작게 유지되므로, 루프에 이 항상 켜진 검색 노드가 있어도 꽤 괜찮은 성능을 볼 수 있습니다. 임베딩 크기, 컨텍스트 제한, 모델 옵션 같은 요소들을 실험하여 어떤 동작을 기대할 수 있는지, 그리고 성능 향상을 위해 어떤 노력이 가치 있는지 확인하는 것이 중요합니다.

<br>

### **Step 4:** 자동 대화 저장

vector store 메모리 유닛이 어떻게 동작해야 하는지 확인했으니, 대화에 새 항목을 추가할 수 있도록 마지막 통합을 수행할 수 있습니다. 바로 스토어 상태를 갱신하기 위해 `add_texts` 메서드를 대신 호출해 주는 runnable입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

########################################################################
## Reset knowledge base and define what it means to add more messages.
convstore = FAISS.from_texts(conversation, embedding=embedder)

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([f"User said {d.get('input')}", f"Agent said {d.get('output')}"])
    return d.get('output')

########################################################################

# instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

chat_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context"
    "\n\nRetrieved Context: {context}"
    "\n\nUser Question: {input}"
    "\nAnswer the user conversationally. Make sure the conversation flows naturally.\n"
    "[Agent]"
)


conv_chain = (
    {
        'context': convstore.as_retriever() | long_reorder | docs2str,
        'input': (lambda x:x)
    }
    | RunnableAssign({'output' : chat_prompt | instruct_llm | StrOutputParser()})
    | partial(save_memory_and_get_output, vstore=convstore)
)

pprint(conv_chain.invoke("I'm glad you agree! I can't wait to get some ice cream there! It's such a good food!"))
print()
pprint(conv_chain.invoke("Can you guess what my favorite food is?"))
print()
pprint(conv_chain.invoke("Actually, my favorite is honey! Not sure where you got that idea?"))
print()
pprint(conv_chain.invoke("I see! Fair enough! Do you know my favorite food now?"))

LLM에 컨텍스트를 주입하는 더 자동화된 전문(full-text) 또는 규칙 기반 접근법과 달리, 이 접근법은 컨텍스트 길이가 통제 불능이 되지 않도록 어느 정도의 통합을 보장합니다. 그 자체로 완벽한 전략은 아니지만, 비정형 대화에서는 뚜렷한 개선이며(심지어 슬롯 채우기를 위한 강력한 지시 튜닝 모델도 필요하지 않습니다).

----

<br>

## **Part 3 [실습]:** 문서 청크 검색을 위한 RAG

앞서 문서 로딩을 탐구했으니, 데이터 청크를 임베딩하고 검색할 수 있다는 아이디어는 아마 놀랍지 않을 것입니다. 그렇지만 문서에 RAG를 적용하는 것은 양날의 검이므로 분명 짚고 넘어갈 가치가 있습니다. 처음에는 잘 동작하는 것처럼 **보일** 수 있지만, 진정으로 신뢰할 수 있는 성능을 위해 최적화하려면 추가적인 주의가 필요합니다. 또한 기본적인 LCEL 역량을 복습할 좋은 기회이기도 하니, 무엇을 할 수 있는지 살펴봅시다!

<br>

### **실습:**

이전 예제에서 [`ArxivLoader`](https://reference.langchain.com/python/langchain-community/document_loaders/arxiv/ArxivLoader)의 도움으로 다음 문법을 사용해 비교적 작은 논문 몇 편을 가져왔던 것을 기억하실 겁니다:

```python
from langchain_community.document_loaders import ArxivLoader

docs = [
    ArxivLoader(query="2205.00445").load(),  ## MRKL
    ArxivLoader(query="2210.03629").load(),  ## ReAct
]
```

지금까지 배운 모든 것을 바탕으로, 사용하고 싶은 논문들을 고르고 그에 대해 이야기할 수 있는 챗봇을 개발하세요!

<br>

꽤 큰 작업이지만, 과정의 ***대부분***에 대한 안내가 아래에 제공됩니다. 안내가 끝날 무렵에는 필요한 퍼즐 조각 대부분이 제공될 것이며, 여러분의 진짜 과제는 이를 통합하여 최종 `retrieval_chain`을 만드는 것입니다. 완료하면 마지막 노트북의 평가 실습에서 이 체인(또는 원하는 변형)을 다시 통합할 준비를 하세요!

<br>

### **Task 1**: 문서 로딩 및 청킹

아래 코드 블록은 RAG 체인에 로드할 기본 논문 몇 편을 제공합니다. 원하는 대로 더 많은 논문을 선택해도 좋지만, 긴 문서는 처리에 더 오래 걸린다는 점에 유의하세요. 단순한 RAG 성능을 개선하는 데 도움이 되도록 몇 가지 단순화 가정과 추가 처리 단계가 포함되어 있습니다:

- "References" 섹션이 있으면 그 앞에서 문서를 잘라냅니다. 이렇게 하면 길고 산만한 경향이 있는 인용 및 부록 섹션을 시스템이 고려하지 않게 됩니다.

- 사용 가능한 문서 목록을 나열하는 청크를 삽입하여, 하나의 청크로 모든 문서의 상위 수준 개요를 제공합니다. 파이프라인이 검색마다 메타데이터를 제공하지 않는 경우 유용한 구성 요소이며, 적절하다면 우선순위가 높은 항목 목록에 포함시킬 수도 있습니다.

- 또한 일반 정보를 제공하기 위해 메타데이터 항목도 삽입합니다. 이상적으로는 메타데이터를 흥미로운 문서 간 청크로 병합한 합성 청크도 있으면 좋습니다.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import ArxivLoader, PyMuPDFLoader
from langchain_community.retrievers import ArxivRetriever
from langchain_core.documents import Document
from IPython.display import display, Markdown
from pathlib import Path

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)    

## TODO: Please pick some papers and add them to the list as you'd like
print("Loading Documents")
paper_ids = [
    # "1706.03762",  ## Attention Is All You Need — transformer foundation
    # "1810.04805",  ## BERT — retrieval/embedding foundation
    # "2103.00020",  ## CLIP — multimodal representation learning
    # "2112.10752",  ## Latent Diffusion — generative modeling milestone
    "2005.11401",  ## Original RAG paper
    "2210.03629",  ## ReAct — reasoning + acting / agent foundation
    # "2310.06825",  ## Mistral 7B — strong modern open LLM
    "2306.05685",  ## Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena

    "2602.06176",  ## Large Language Model Reasoning Failures — Feb 2026 survey
    # "2601.01828",  ## Emergent Introspective Awareness in Large Language Models
    "2603.18272",  ## Retrieval-Augmented LLM Agents: Learning to Learn from 
]
try:
    docs = [doc[0] for doc in ArxivRetriever(get_full_documents=True, doc_content_chars_max=1000000).batch(paper_ids, config={"max_concurrency": 1})]
except Exception as error:
    cached_paper = Path("cached_papers/2210.03629v3.pdf")
    if not cached_paper.is_file():
        raise RuntimeError("Live arXiv retrieval failed and the cached ReAct paper is missing.") from error
    print("Live arXiv retrieval was unavailable; loading a cached ReAct paper to keep the workflow available.")
    cached_pages = PyMuPDFLoader(str(cached_paper)).load()
    docs = [Document(
        page_content="\n\n".join(page.page_content for page in cached_pages),
        metadata={"Title": "ReAct: Synergizing Reasoning and Acting in Language Models", "entry_id": "2210.03629v3"},
    )]

## Cut the paper short if references is included.
## This is a standard string in papers.
for doc in docs:
    content = doc.page_content
    if "References" in content:
        doc.page_content = content[:content.index("References")]

## Split the documents and also filter out stubs (overly short chunks)
print("Chunking Documents")
docs_chunks = [text_splitter.split_documents([doc]) for doc in docs]
docs_chunks = [[c for c in dchunks if len(c.page_content) > 200] for dchunks in docs_chunks]

## Make some custom Chunks to give big-picture details
doc_string = "Available Documents:"
doc_metadata = []
for chunks in docs_chunks:
    metadata = getattr(chunks[0], 'metadata', {})
    doc_string += "\n - " + metadata.get('Title')
    doc_metadata += [str(metadata)]

extra_chunks = [doc_string] + doc_metadata

## Printing out some summary information for reference
pprint(doc_string, '\n')
for i, chunks in enumerate(docs_chunks):
    print(f"Document {i}")
    print(f" - # Chunks: {len(chunks)}")
    print(f" - Metadata: ")
    pprint(chunks[0].metadata)
    display(Markdown(f"<details><summary>Chunk 0</summary>{chunks}</details>"))
    print()

<br>

### **Task 2**: 문서 Vector Store 구성하기

모든 구성 요소를 갖추었으니, 이를 둘러싼 인덱스를 만들어 봅시다:

In [ ]:
%%time
print("Constructing Vector Stores")
vecstores = [FAISS.from_texts(extra_chunks, embedder)]
vecstores += [FAISS.from_documents(doc_chunks, embedder) for doc_chunks in docs_chunks]

<br>

거기서부터 다음 유틸리티를 사용해 인덱스들을 하나로 합칠 수 있습니다:

In [ ]:
from faiss import IndexFlatL2
from langchain_community.docstore.in_memory import InMemoryDocstore

embed_dims = len(embedder.embed_query("test"))
def default_FAISS():
    '''Useful utility for making an empty FAISS vectorstore'''
    return FAISS(
        embedding_function=embedder,
        index=IndexFlatL2(embed_dims),
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
        normalize_L2=False
    )

def aggregate_vstores(vectorstores):
    ## Initialize an empty FAISS Index and merge others into it
    ## We'll use default_faiss for simplicity, though it's tied to your embedder by reference
    agg_vstore = default_FAISS()
    for vstore in vectorstores:
        agg_vstore.merge_from(vstore)
    return agg_vstore

## Unintuitive optimization; merge_from seems to optimize constituent vector stores away
docstore = aggregate_vstores(vecstores)

print(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")

<br>

### **Task 3: [실습]** RAG Chain 구현하기

드디어 RAG 파이프라인을 구현하기 위한 모든 퍼즐 조각이 갖춰졌습니다! 복습하자면, 이제 우리에게는 다음이 있습니다:

- 대화 메모리를 위한 vector store를 처음부터 구성하는 방법(그리고 `default_FAISS()`로 빈 스토어를 초기화하는 방법)

- `ArxivLoader` 유틸리티로 얻은 유용한 문서 정보가 미리 로드된 vector store(`docstore`에 저장됨).

몇 가지 유틸리티의 도움을 더 받으면 드디어 체인을 통합할 준비가 됩니다! 몇 가지 추가 편의 유틸리티(`doc2str`과 이제는 익숙한 `RPrint`)가 제공되지만 사용은 선택입니다. 또한 시작용 프롬프트와 구조도 정의되어 있습니다.

> **이 모든 것을 바탕으로:** `retrieval_chain`을 구현하세요.

In [ ]:
from langchain_community.document_transformers import LongContextReorder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from langchain_nvidia_ai_endpoints import ChatNVIDIA

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import gradio as gr
from functools import partial
from operator import itemgetter

instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

convstore = default_FAISS()

def save_memory_and_get_output(d, vstore):
    """Accepts 'input'/'output' dictionary and saves to convstore"""
    vstore.add_texts([
        f"User previously responded with {d.get('input')}",
        f"Agent previously responded with {d.get('output')}"
    ])
    return d.get('output')

initial_msg = (
    "Hello! I am a document chat agent here to help the user!"
    f" I have access to the following documents: {doc_string}\n\nHow can I help you?"
)

chat_prompt = ChatPromptTemplate.from_messages([("system",
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked: {input}\n\n"
    " From this, we have retrieved the following potentially-useful info: "
    " Conversation History Retrieval:\n{history}\n\n"
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational.)"
), ('user', '{input}')])

stream_chain = chat_prompt| RPrint() | instruct_llm | StrOutputParser()

################################################################################################
## BEGIN TODO: Implement the retrieval chain to make your system work!

retrieval_chain = (
    {'input' : (lambda x: x)}
    ## TODO: Make sure to retrieve history & context from convstore & docstore, respectively.
    ## HINT: Our solution uses RunnableAssign, itemgetter, long_reorder, and docs2str
    | RunnableAssign({'history' : lambda d: None})
    | RunnableAssign({'context' : lambda d: None})
)

## END TODO
################################################################################################

def chat_gen(message, history=[], return_buffer=True):
    buffer = ""
    ## First perform the retrieval based on the input message
    retrieval = retrieval_chain.invoke(message)
    line_buffer = ""

    ## Then, stream the results of the stream_chain
    for token in stream_chain.stream(retrieval):
        buffer += token
        ## If you're using standard print, keep line from getting too long
        yield buffer if return_buffer else token

    ## Lastly, save the chat exchange to the conversation memory buffer
    save_memory_and_get_output({'input':  message, 'output': buffer}, convstore)


## Start of Agent Event Loop
test_question = "Tell me about RAG!"  ## <- modify as desired

## Before you launch your gradio interface, make sure your thing works
for response in chat_gen(test_question, return_buffer=False):
    print(response, end='')

### **Task 4:** Gradio 챗봇과 상호작용하기

In [ ]:
# chatbot = gr.Chatbot(value = [{"role": "user", "content": initial_msg}])
# demo = gr.ChatInterface(chat_gen, chatbot=chatbot).queue()

# try:
#     demo.launch(debug=True, share=True, show_api=False)
#     demo.close()
# except Exception as e:
#     demo.close()
#     print(e)
#     raise e

<br>

----

<br>

## **Part 4:** 평가를 위해 인덱스 저장하기

RAG 체인을 구현한 뒤에는 [공식 문서](https://python.langchain.com/docs/integrations/vectorstores/faiss#saving-and-loading)에 나온 대로 축적된 vector store를 저장하세요. 최종 평가에서 다시 사용할 기회가 있습니다!

In [ ]:
## Save and compress your index
docstore.save_local("docstore_index")
!tar czvf docstore_index.tgz docstore_index

!rm -rf docstore_index

모든 것이 제대로 저장되었다면, (pip 요구 사항이 설치되어 있다고 가정할 때) 다음 줄을 호출하여 압축된 `tgz` 파일에서 인덱스를 가져올 수 있습니다. 셀이 인덱스를 가져올 수 있음을 확인한 뒤, 마지막 노트북에서 사용할 수 있도록 `docstore_index.tgz`를 다운로드하세요!

In [ ]:
from langchain_community.vectorstores import FAISS

!tar xzvf docstore_index.tgz
new_db = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
docs = new_db.similarity_search("Testing the index")
print(docs[0].page_content[:1000])

-----

<br>

## **Part 5:** 마무리

축하합니다! RAG 체인이 잘 동작한다면 이제 **RAG 평가 [Assessment]** 섹션으로 넘어갈 준비가 되었습니다!

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>